# Карта курса

Курс устроен как одна стройка, а не как шесть отдельных задач. В первой работе
мы возьмём девять выгруженных файлов интернет-магазина и сделаем из них отчёт
по дням. В последней – обучим модель на данных, которые прошли проверку
качества, легли в таблицы с понятной структурой и собираются по расписанию.
Между этими точками – четыре работы, каждая из которых чинит поломку,
оставшуюся от предыдущей.

Эта тетрадь короткая и без заданий: она нужна, чтобы вы один раз увидели
маршрут целиком и потом не гадали, зачем очередной слой.

## Несколько слов, которые будут встречаться весь курс

Полный словарь не нужен – эти восемь понятий достаточно просто узнавать
в тексте. Остальные термины появятся там, где возникнет задача, которую они
решают.

| Термин | Простыми словами |
|---|---|
| **Инженерия данных – Data Engineering** | Работа над тем, чтобы данные доехали до потребителя вовремя, целиком и без искажений. |
| **Источник – source** | Система, из которой мы забираем данные: база приложения, файлы партнёра, внешний сервис. |
| **Конвейер обработки – data pipeline** | Цепочка шагов от источника до готового результата, которую можно запустить повторно. |
| **Хранилище – storage** | Место, где данные лежат между шагами: файлы, база, аналитический движок. |
| **Пакетная обработка – batch processing** | Ждём, пока накопится порция данных, и обрабатываем её целиком. |
| **Потоковая обработка – streaming** | Обрабатываем данные по мере появления, не дожидаясь пачки. |
| **Хранилище данных – Data Warehouse, DWH** | Общие для компании таблицы, собранные под вопросы бизнеса, а не под работу приложения. |
| **Потребитель данных – data consumer** | Тот, ради кого всё строится: отчёт, дашборд или модель машинного обучения. |

## Подготовка

Ячейка ниже готовит окружение и данные. Её достаточно запустить.

In [ ]:
# Готовим окружение: в VS Code и Jupyter берём открытую папку курса,
# в Colab материалы скачиваются один раз. Всё остальное – в src/notebook_env.py.
from pathlib import Path
import subprocess, sys

ROOT = next((c for s in (Path.cwd().resolve(), Path("/content/data-engineering-ai-course"))
             for c in (s, *s.parents) if (c / ".course-root").exists()), None)
if ROOT is None:                                   # чистая сессия Colab
    url = ''
    if not url:
        raise RuntimeError("Откройте папку курса в VS Code/Jupyter или склонируйте "
                           "репозиторий в /content/data-engineering-ai-course")
    subprocess.run(["git", "clone", "--depth", "1", url,
                    "/content/data-engineering-ai-course"], check=True)
    ROOT = Path("/content/data-engineering-ai-course")
sys.path[:0] = [str(ROOT), str(ROOT / "src")]
from notebook_env import setup_course
setup_course(ROOT)


In [ ]:
import pandas as pd
import paths
from course_visuals import show_visual
from get_data import get_olist, source_kind

RAW = get_olist()
print("набор данных:", source_kind())

## Что мы строим

In [ ]:
show_visual("course")

Слева – девять файлов выгрузки, справа – отчёты и модель. Всё, что между ними,
и есть предмет курса.

Каждый следующий слой появляется не потому, что «так принято», а потому что без
него предыдущий перестаёт работать:

| Работа | Что появляется | Что ломается сразу после |
|---|---|---|
| 1 | приём файлов и первый расчёт на pandas | всё читается в память целиком, типы теряются при первой же записи |
| 2 | колоночное хранение и SQL | считаем быстро, но считаем по данным, которые никто не проверял |
| 3 | схема, правила качества, отбраковка | данные чистые, но каждый аналитик собирает свой отчёт и получает своё число |
| 4 | общая модель данных и витрины | витрины отличные, но пересобираются руками и в правильном порядке |
| 5 | расписание, повторные запуски, восстановление истории | данных стало в сто раз больше, а часть из них нужна не завтра, а через секунду |
| 6 | рост объёмов, поток событий, признаки для модели | – |

Правая колонка – это и есть содержание курса. Инженер данных занят не столько
написанием нового кода, сколько починкой того, что сломалось от роста объёма,
смены требований или появления нового источника.

## Что курс даёт и чего не даёт

К концу курса вы должны уметь взять описание чужой системы и ответить:
откуда придут данные, что можно считать ночью, а чему нужен ответ за секунды,
где ставить проверки, что означает одна строка в каждой таблице и что
произойдёт при повторном запуске обработки. Это рассуждение важнее знания
конкретных инструментов: инструменты меняются каждые несколько лет, вопросы
остаются те же.

Кластеры мы поднимать не будем. Airflow, Kafka и Spark появятся как ответы на
конкретные затруднения, вместе с описанием того, когда они не нужны. Вместо них
возьмём небольшие рабочие модели: планировщик задач на две сотни строк, журнал
событий в памяти, аналитический движок внутри процесса. На них видна механика,
а не интерфейс конкретного продукта.

Слова «архитектура» в отрыве от цифр в курсе не будет. Каждое утверждение вида
«здесь хватит обычной базы» опирается на объём данных, допустимую задержку и
цену ошибки – спрашивать будут ровно об этом. Потренироваться на чужих
системах можно на разборах в папке `architecture_cases`.

## Как устроена каждая работа

Сначала – короткий словарь новых слов, чтобы термины не мешали читать. Затем
ситуация, в которой текущая система перестаёт справляться, и разбор того, что
происходит: рабочий код, схема, небольшие опыты – что будет, если убрать
проверку, перезапустить обработку дважды, соединить таблицы неправильно. Дальше
задание, где нужно принять решение, и автоматическая проверка результата.
В конце – та же задача в другой отрасли, где ответ оказывается иным.

Проверки смотрят на свойства результата: сходится ли количество заказов, не
потерялись ли строки, одинаков ли результат при повторном запуске. Запустить всё
подряд и получить зачёт не выйдет, но и угадывать «что имел в виду
преподаватель» не придётся.

## Данные, на которых всё построено

In [ ]:
files = sorted(RAW.glob("*.csv"))
pd.DataFrame({"файл": [f.name for f in files],
              "МБ": [round(f.stat().st_size / 1024 ** 2, 2) for f in files]})

Это выгрузка бразильского маркетплейса Olist: заказы, их позиции, платежи,
отзывы, покупатели, продавцы, товары и география. Данные настоящие, со всеми
свойственными им странностями – этим они и ценны.

Если в строке «набор данных» указано `synthetic`, значит доступа к Kaggle не
было и курс работает на офлайн-наборе с такой же структурой. Задания и проверки
рассчитаны на оба варианта, отличаться будут только числа.

In [ ]:
orders = pd.read_csv(RAW / "olist_orders_dataset.csv")
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW / "olist_order_payments_dataset.csv")

print(f"строк в таблице заказов   : {len(orders):>8,}")
print(f"строк в позициях заказов  : {len(items):>8,}")
print(f"строк в платежах          : {len(payments):>8,}")

Три числа описывают одни и те же покупки, и все три разные. У заказа бывает
несколько товарных позиций и несколько платёжных операций: часть суммы картой,
часть – подарочным сертификатом. Пока таблицы лежат отдельно, это никому не
мешает. Стоит соединить их в одном запросе – и выручка вырастет на пустом месте,
потому что каждая сумма посчитается столько раз, сколько строк оказалось рядом.

С разбора этой ошибки начинается первая работа.

## Порядок работ

1. `01_data_flow` – откуда берутся данные и что означает одна строка таблицы
2. `02_storage_sql` – где их хранить и как считать на SQL
3. `03_etl_quality` – как отличить дефект данных от нормы
4. `04_dwh_modeling` – как собрать модель данных и витрины
5. `05_pipelines` – как запускать всё это без ручного участия
6. `06_scale_streaming_ai` – что меняется при росте объёма и как данные доходят до модели

Работы связаны файлами в папке `data`, поэтому порядок обязателен. Если открыть
четвёртую, не выполнив третью, вы получите не загадочную ошибку, а сообщение
о том, какого файла не хватает и где он создаётся.

Состояние своих работ можно посмотреть командой `python course_check.py`.